# Netflix ANN Movie Recommendation System

This ANN is trained to recommend Netflix titles using **Mood + Rating + Year** preferences. The title/name is the final recommendation output.

In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score

C:\Users\Prachi\AppData\Local\Temp\ipykernel_22568\2898320029.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
dataset = pd.read_csv("netflix_titles.csv")
dataset.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,"August 14, 2020",2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,"December 23, 2016",2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,"December 20, 2018",2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow..."
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,"November 16, 2017",2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi..."
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,"January 1, 2020",2008,PG-13,123 min,Dramas,A brilliant group of students become card-coun...


In [3]:
dataset["type"].value_counts()

type
Movie      5377
TV Show    2410
Name: count, dtype: int64

## 1. Create Mood Categories

The dataset does not contain a separate `mood` column. Mood is inferred from the `listed_in` genre metadata.

In [4]:
MOODS = {
    "Action": ["action", "adventure", "martial arts", "sports"],
    "Comedy": ["comedies", "comedy", "stand-up", "humor"],
    "Drama": ["dramas", "drama", "independent movies"],
    "Romance": ["romantic", "romance"],
    "Thriller": ["thrillers", "thriller", "crime", "mysteries"],
    "Horror": ["horror", "supernatural"],
    "Family": ["children", "kids", "family"],
    "Documentary": ["documentaries", "documentary"],
    "Feel-Good": ["comedies", "romantic", "family", "music", "musicals"],
    "Adventure": ["adventure", "action", "fantasy", "sci-fi"]
}
moods = list(MOODS)

def mood_vector(listed_in):
    text = str(listed_in).lower()
    return [float(any(k in text for k in kws)) for kws in MOODS.values()]

## 2. Feature Engineering

In [5]:
data = dataset.copy()
data["year"] = pd.to_numeric(data["release_year"], errors="coerce").fillna(0)

data["genre_count"] = data["listed_in"].fillna("").astype(str).apply(
    lambda x: len([v for v in x.split(",") if v.strip()])
)
data["cast_count"] = data["cast"].fillna("").astype(str).apply(
    lambda x: len([v for v in x.split(",") if v.strip()])
)
data["country_count"] = data["country"].fillna("").astype(str).apply(
    lambda x: len([v for v in x.split(",") if v.strip()])
)
data["description_length"] = data["description"].fillna("").astype(str).str.len()
data["director_available"] = data["director"].fillna("").astype(str).str.strip().ne("").astype(float)

ratings = sorted(data["rating"].fillna("Unknown").astype(str).unique().tolist())
rating_to_num = {r:i for i,r in enumerate(ratings)}

def candidate_features(row):
    rating = str(row["rating"]) if pd.notna(row["rating"]) else "Unknown"
    rv = np.zeros(len(ratings))
    rv[rating_to_num.get(rating, 0)] = 1
    return np.r_[
        float(row["year"]), float(row["genre_count"]), float(row["cast_count"]),
        float(row["country_count"]), float(row["description_length"]),
        float(row["director_available"]), rv, mood_vector(row["listed_in"])
    ]

## 3. Create Training Examples

Because the Netflix dataset does not contain a user preference/relevance label, training targets are generated from year, rating and mood matching.

In [6]:
rng = np.random.default_rng(123)
n_samples = 60000
candidate_idx = rng.integers(0, len(data), size=n_samples)
query_year = rng.integers(2000, 2027, size=n_samples)
query_rating = rng.choice(ratings, size=n_samples)
query_mood = rng.integers(0, len(moods), size=n_samples)

candidate_matrix = np.vstack([candidate_features(r) for _, r in data.iterrows()])

X_list, y_list = [], []

for i, idx in enumerate(candidate_idx):
    row = data.iloc[idx]
    cf = candidate_matrix[idx]

    q_rating = np.zeros(len(ratings))
    q_rating[rating_to_num.get(query_rating[i], 0)] = 1

    q_mood = np.zeros(len(moods))
    q_mood[query_mood[i]] = 1

    year_match = max(0, 1 - abs(float(row["year"]) - query_year[i]) / 30)
    rating_match = float(str(row["rating"]) == query_rating[i])
    mood_match = float(np.array(mood_vector(row["listed_in"]))[query_mood[i]])

    score = 0.50*mood_match + 0.30*rating_match + 0.20*year_match

    X_list.append(np.r_[cf, query_year[i], q_rating, q_mood])
    y_list.append(score)

X = np.asarray(X_list)
y = np.asarray(y_list)

print("Training samples:", X.shape[0])
print("Input features:", X.shape[1])

Training samples: 60000
Input features: 57


In [7]:
train_X, test_X, train_y, test_y = train_test_split(
    X, y, test_size=0.30, random_state=123
)

sc = StandardScaler()
train_X = sc.fit_transform(train_X)
test_X = sc.transform(test_X)

## 4. ANN Model

In [8]:
model = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32, 16),
    activation="relu",
    solver="adam",
    batch_size=32,
    max_iter=80,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=8,
    random_state=123
)

model.fit(train_X, train_y)

MLPRegressor(batch_size=32, early_stopping=True,
             hidden_layer_sizes=(128, 64, 32, 16), max_iter=80,
             n_iter_no_change=8, random_state=123)

In [10]:
predictions = model.predict(test_X)
print("MAE:", mean_absolute_error(test_y, predictions))
print("R2 Score:", r2_score(test_y, predictions))

MAE: 0.003434435545826964
R2 Score: 0.9967398908276607


## 5. Save ANN and Preprocessing

In [11]:
joblib.dump(model, "netflix_ann_recommender.pkl")
joblib.dump(sc, "netflix_recommender_scaler.pkl")
joblib.dump(ratings, "netflix_ratings.pkl")
joblib.dump(moods, "netflix_moods.pkl")
print("All ANN recommender files saved.")

All ANN recommender files saved.
